# HoloKai GLiNER — Option B annotation experiment

Copies the Final Run model, dependencies, training parameters and exact inclusive
token-span/label micro P/R/F1 evaluation. Only the train/validation paths and training
output directory change in the JSON configuration. All results live under
`/content/drive/MyDrive/Holokai_Project/week8/Final_model_Option_B/`.

**Data status:** Option B additions are machine-generated `AUTOMATED_UNVERIFIED`
candidate annotations, not adjudicated ground truth. The original 11 labels, tokens
and existing annotations are retained. Test annotations also changed: differences
from the previous experiment's scores do not isolate model improvement.

Upload this notebook, `configs/finetune_final_run_option_b.json` and
`option_b_colab_support.zip` into the experiment folder, retaining the `configs/`
subfolder. Identical shared project scripts are reused; otherwise the original
scripts are unpacked into an isolated `support/` folder without overwriting files.
The original notebook, config, datasets and results are preserved.

Training and optional validation analyses default to **True**, matching the original
code. V4 execution defaults to **False** and is enabled in Step 10 after model and
validation choices are frozen. Creating this notebook does not execute any cells.


## Step 1 — Environment setup and dependency preflight


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

assert torch.cuda.is_available(), "Enable a GPU runtime before continuing."


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

%cd "/content/drive/MyDrive/Holokai_Project"


In [ ]:
from pathlib import Path
from collections import Counter
import hashlib
import importlib
import json
import subprocess
import sys

PROJECT = Path("/content/drive/MyDrive/Holokai_Project")
DATA_ROOT = PROJECT / "week8/Holokai_Option_B_Annotation_Files"
RUN_ROOT = PROJECT / "week8/Final_model_Option_B"
TRAIN_DATA = DATA_ROOT / "train_gliner_mapping_option_B.json"
VAL_DATA = DATA_ROOT / "val_gliner_mapping_option_B.json"
TEST_DATA = DATA_ROOT / "corpus_V4_gliner_mapping_option_B.json"
RUN_CONFIG = RUN_ROOT / "configs/finetune_final_run_option_b.json"
TRAIN_OUT = RUN_ROOT / "training"
BASELINE_OUT = RUN_ROOT / "baseline"
MODEL_EVAL_OUT = RUN_ROOT / "model_eval"
COMPARISON_OUT = RUN_ROOT / "comparison"
THRESHOLD_OUT = RUN_ROOT / "threshold_tuning"
BEST_MODEL_NAME = "best_model"
BEST_MODEL_PATH = TRAIN_OUT / BEST_MODEL_NAME
BASE_MODEL = "knowledgator/gliner-pii-base-v1.0"
MAPPED_SCHEMA = "recommended"
REFERENCE_CONFIG = {'label_schema': 'recommended', 'base_model': 'knowledgator/gliner-pii-base-v1.0', 'train_data': '/content/drive/MyDrive/Holokai_Project/finetune/data/train_gliner_mapping.json', 'val_data': '/content/drive/MyDrive/Holokai_Project/finetune/data/val_gliner_mapping.json', 'output_dir': '/content/drive/MyDrive/Holokai_Project/final_model/outputs/training', 'learning_rate': 0.00012, 'batch_size': 16, 'eval_batch_size': 8, 'epochs': 8, 'warmup_ratio': 0.1, 'weight_decay': 0.1, 'seed': 42, 'eval_steps': 100, 'logging_steps': 10, 'threshold': 0.4, 'flat_ner': False, 'gradient_accumulation_steps': 1, 'max_grad_norm': 1.0, 'adam_beta1': 0.9, 'adam_beta2': 0.999, 'adam_epsilon': 1e-08, 'lr_scheduler_type': 'linear', 'loss_reduction': 'mean', 'device': 'auto', 'save_best_model': True}
SUPPORT_HASHES = {'finetune/requirements-colab.txt': '8109cb2ec9815906b0e178f9687f7aca6eab59b5573df33524497a1c3f2ce3e8', 'finetune/scripts/train_gliner_final_run.py': '6c38397ff4585b6c245d77786f4a0f851a4c7bc7059ab636bdc1e61772d3b107', 'finetune/scripts/evaluate_gliner.py': 'dbf305c3cc14a149291cb2715804a6ea56b3ee4072cd8575266f623ed1fe1300', 'finetune/scripts/label_schemas.py': '871d28532ec98e4130ae7e5d0b4f34c2ea1e7b8b63b5455da6fbf3d76fcb7b47', 'finetune/scripts/validate_gliner_data.py': 'c6d0536f8692199da3b42d6b4f156481f3f6b1f0ea53386364ed006657021667', 'finetune/scripts/tune_per_label_thresholds.py': '9b6f092508bf9a5b0ec81f35c885f6f39f2049eb2fd22c3679cb01f508fca06b'}

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def resolve_support():
    """Reuse identical scripts; otherwise extract an isolated, verified copy."""
    import zipfile
    def matches(root):
        return all((root / name).is_file() and sha256_file(root / name) == digest
                   for name, digest in SUPPORT_HASHES.items())
    if matches(PROJECT):
        return PROJECT
    support = RUN_ROOT / "support"
    if matches(support):
        return support
    archive_path = RUN_ROOT / "option_b_colab_support.zip"
    if not archive_path.is_file():
        raise FileNotFoundError(
            f"Shared scripts are missing or differ from the original run. Upload "
            f"option_b_colab_support.zip to {RUN_ROOT}; original files will be preserved.")
    with zipfile.ZipFile(archive_path) as archive:
        for name, digest in SUPPORT_HASHES.items():
            raw = archive.read(name)
            if hashlib.sha256(raw).hexdigest() != digest:
                raise ValueError(f"Support archive checksum mismatch: {name}")
            target = support / name
            if target.exists():
                if not target.is_file() or sha256_file(target) != digest:
                    raise FileExistsError(f"Incompatible support file preserved: {target}")
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with target.open("xb") as handle:
                    handle.write(raw)
    assert matches(support)
    return support

def import_run_modules(support):
    scripts = support / "finetune/scripts"
    sys.path.insert(0, str(scripts))
    names = ("label_schemas", "validate_gliner_data", "evaluate_gliner",
             "train_gliner_final_run", "tune_per_label_thresholds")
    # Invalidate the entire import group if one member belongs to another run.
    if any(name in sys.modules and Path(sys.modules[name].__file__).resolve() !=
           (scripts / f"{name}.py").resolve() for name in names):
        for name in names:
            sys.modules.pop(name, None)
    modules = {name: importlib.import_module(name) for name in names}
    for name, module in modules.items():
        if Path(module.__file__).resolve() != (scripts / f"{name}.py").resolve():
            raise ImportError(f"Unexpected script loaded: {module.__file__}")
    return scripts, modules

def load_json(path):
    from validate_gliner_data import unique_keys, reject_constant
    return json.loads(Path(path).read_text(encoding="utf-8"),
                      object_pairs_hook=unique_keys, parse_constant=reject_constant)

def preserve_json(path, value):
    """Reuse an identical result, never replace an existing one."""
    path = Path(path)
    if path.exists():
        if load_json(path) != value:
            raise FileExistsError(f"Different saved result preserved: {path}")
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("x", encoding="utf-8") as handle:
        json.dump(value, handle, indent=2, ensure_ascii=False, allow_nan=False)
        handle.write("\n")

def preserve_csv(path, frame):
    import pandas as pd
    path = Path(path)
    if path.exists():
        pd.testing.assert_frame_equal(pd.read_csv(path), frame.reset_index(drop=True),
                                      check_dtype=False)
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False, mode="x")

def check_result_files(output):
    files = [Path(output) / name for name in
             ("metrics.json", "per_label_metrics.csv", "predictions.json")]
    if any(path.exists() for path in files):
        if not all(path.is_file() for path in files):
            raise FileExistsError(f"Partial results preserved at {output}; use a new directory.")
        return True
    return False

def expected_evaluation(model_name, data_path, threshold):
    from evaluate_gliner import load_data
    rows, digest = load_data(Path(data_path), MAPPED_SCHEMA)
    return {"model": str(model_name), "data_sha256": digest,
            "sample_count": len(rows), "label_schema": MAPPED_SCHEMA,
            "batch_size": 8, "flat_ner": False, "multi_label": False,
            "threshold": threshold, "averaging": "micro",
            "matching": "exact inclusive token start/end + exact label within sample"}

def verify_evaluation(output, expected):
    from label_schemas import get_labels
    if not check_result_files(output):
        raise FileNotFoundError(f"Evaluation results missing: {output}")
    result = load_json(Path(output) / "metrics.json")
    for key, value in expected.items():
        if result.get("metadata", {}).get(key) != value:
            raise ValueError(f"Incompatible {key} in {output}; existing files preserved.")
    if set(result["per_label"]) != set(get_labels(MAPPED_SCHEMA)):
        raise ValueError(f"Unexpected result labels: {output}")
    return result

def evaluate_fixed_or_load(model_name, data_path, output):
    expected = expected_evaluation(model_name, data_path, 0.40)
    if not check_result_files(output):
        subprocess.run([sys.executable, str(SCRIPT_ROOT / "evaluate_gliner.py"),
                        "--label-schema", MAPPED_SCHEMA, "--model", str(model_name),
                        "--data", str(data_path), "--threshold", "0.40", "--batch-size", "8",
                        "--output-dir", str(output), "--device", "auto"], check=True)
    result = verify_evaluation(output, expected)
    print("Results available:", output)
    return result

def validate_frozen_thresholds(path, model_path, validation_path, test_path):
    from evaluate_gliner import load_selected_thresholds
    selected, candidate, digest, metadata = load_selected_thresholds(
        path, MAPPED_SCHEMA, False, str(model_path))
    if metadata.get("data_sha256") != sha256_file(validation_path):
        raise ValueError("Thresholds were not selected on the current Option B validation file.")
    if metadata["data_sha256"] == sha256_file(test_path):
        raise ValueError("Test data cannot be the threshold-selection data.")
    return selected, candidate, digest, metadata

def display_result(title, result):
    import pandas as pd
    from IPython.display import display
    fields = ["precision", "recall", "f1", "tp", "fp", "fn", "support"
    fmt = {name: "{:.2%}" for name in ("precision", "recall", "f1")}
    print(title)
    display(pd.DataFrame([result["overall"]])[fields].style.format(fmt))
    table = pd.DataFrame.from_dict(result["per_label"], orient="index")[fields]
    table.index.name = "label"
    table.insert(0, "threshold", [result.get("thresholds", {}).get(label, 0.40)
                                  for label in table.index])
    display(table.style.format(fmt))

def compare_results(left, right, output, left_name, right_name, left_path, right_path):
    import pandas as pd
    from IPython.display import display
    for key in ("data_sha256", "sample_count", "label_schema", "batch_size", "flat_ner",
                "multi_label", "matching", "averaging"):
        if left["metadata"][key] != right["metadata"][key]:
            raise ValueError(f"Comparison evaluation mismatch: {key}")
    if left["overall"]["support"] != right["overall"]["support"]:
        raise ValueError("Comparison gold support mismatch")
    fields = ["precision", "recall", "f1", "tp", "fp", "fn", "support"]
    overall = [{"metric": field, left_name: left["overall"][field],
                right_name: right["overall"][field],
                "change_right_minus_left": right["overall"][field] - left["overall"][field]}
               for field in fields]
    per_label = []
    for label in left["per_label"]:
        row = {"label": label}
        if left["per_label"][label]["support"] != right["per_label"][label]["support"]:
            raise ValueError(f"Comparison per-label support mismatch: {label}")
        for field in fields:
            a, b = left["per_label"][label][field], right["per_label"][label][field]
            row.update({f"{field}_{left_name}": a, f"{field}_{right_name}": b,
                        f"{field}_delta": b - a})
        per_label.append(row)
    record = {"left_result": str(left_path), "right_result": str(right_path),
              "difference": f"{right_name} - {left_name}",
              "data_sha256": left["metadata"]["data_sha256"],
              "thresholds_sha256": right["metadata"].get("thresholds_sha256"),
              "comparison": overall, "per_label_comparison": per_label}
    output = Path(output)
    preserve_json(output / "overall_comparison.json", record)
    preserve_csv(output / "overall_comparison.csv", pd.DataFrame(overall))
    preserve_csv(output / "per_label_comparison.csv", pd.DataFrame(per_label))
    print(f"Change = {right_name} - {left_name}; rate changes are absolute proportions.")
    display(pd.DataFrame(overall))
    display(pd.DataFrame(per_label))
    print("Comparison saved:", output)


SUPPORT_ROOT = resolve_support()
requirements = SUPPORT_ROOT / "finetune/requirements-colab.txt"
if not RUN_CONFIG.is_file():
    raise FileNotFoundError(f"Upload configs/finetune_final_run_option_b.json to {RUN_ROOT}")
print("Support root:", SUPPORT_ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(requirements)], check=True)


## Step 2 — Validate Option B mapped data without rewriting it

Validate all three files with the existing evaluator, show hashes and label counts,
and reject duplicate token sequences or cross-split token overlap. This replaces
the old original-to-mapped entity-count equality check: Option B intentionally adds
entities. No old unmapped files are required. Test inspection here is structural;
test metrics never select a model or a threshold.


In [ ]:
SCRIPT_ROOT, modules = import_run_modules(SUPPORT_ROOT)
evaluator = modules["evaluate_gliner"]
tuner = modules["tune_per_label_thresholds"]
LABEL_MAPPING = dict(modules["label_schemas"].RECOMMENDED_LABEL_MAPPING)
MAPPED_LABELS = list(modules["label_schemas"].get_labels(MAPPED_SCHEMA))
EXPECTED_COUNTS = {"train": (1509, 4578), "validation": (171, 510), "test": (648, 1413)}
data_report, token_sets = {}, {}
for split, path in (("train", TRAIN_DATA), ("validation", VAL_DATA), ("test", TEST_DATA)):
    rows, digest = evaluator.load_data(path, MAPPED_SCHEMA)
    counts = Counter(entity[2] for row in rows for entity in row["ner"])
    tokens = [tuple(row["tokenized_text"]) for row in rows]
    if len(set(tokens)) != len(tokens):
        raise ValueError(f"Duplicate token sequences in {split}")
    if set(counts) != set(MAPPED_LABELS):
        raise ValueError(f"{split} must cover all 11 recommended labels")
    observed = (len(rows), sum(counts.values()))
    if observed != EXPECTED_COUNTS[split]:
        raise ValueError(f"Option B data changed: {split} {observed} != {EXPECTED_COUNTS[split]}")
    token_sets[split] = set(tokens)
    data_report[split] = {"path": str(path), "samples": len(rows),
                          "entities": sum(counts.values()), "sha256": digest,
                          "per_label": {label: counts[label] for label in MAPPED_LABELS}}
for left, right in (("train", "validation"), ("train", "test"), ("validation", "test")):
    if token_sets[left] & token_sets[right]:
        raise ValueError(f"Token sequence overlap: {left}/{right}")
print(json.dumps(data_report, indent=2))
del rows, tokens, token_sets


## Step 3 — Configuration parity and execution switches

The embedded reference is an exact copy of the original Final Run JSON. Except for
`train_data`, `val_data`, and `output_dir`, every field must match. Training starts
from the same pretrained model, never the old best model. A fresh run uses
`RESUME_FROM_CHECKPOINT=None`; interrupted training may resume using `"latest"`
from this experiment only. Configuration checking never loads or trains a model.


In [ ]:
TRAIN_SCRIPT = SCRIPT_ROOT / "train_gliner_final_run.py"
cfg = modules["train_gliner_final_run"].load_config(RUN_CONFIG)
path_fields = {"train_data", "val_data", "output_dir"}
assert set(cfg) == set(REFERENCE_CONFIG), "Configuration keys changed"
assert {k: v for k, v in cfg.items() if k not in path_fields} == {
    k: v for k, v in REFERENCE_CONFIG.items() if k not in path_fields
}, "Model or training parameters differ from the original Final Run"
assert cfg["train_data"] == str(TRAIN_DATA)
assert cfg["val_data"] == str(VAL_DATA)
assert cfg["output_dir"] == str(TRAIN_OUT)
RUN_TRAINING = True
RUN_OPTIONAL_ANALYSES = True
RUN_SAVED_VALIDATION = True
RESUME_FROM_CHECKPOINT = None
if RESUME_FROM_CHECKPOINT not in (None, "latest"):
    checkpoint = Path(RESUME_FROM_CHECKPOINT).resolve()
    if checkpoint.parent != TRAIN_OUT.resolve():
        raise ValueError("Only this experiment's checkpoint may be resumed")
print(json.dumps(cfg, indent=2))
print("Execution switches:", {"RUN_TRAINING": RUN_TRAINING,
      "RUN_OPTIONAL_ANALYSES": RUN_OPTIONAL_ANALYSES,
      "RUN_SAVED_VALIDATION": RUN_SAVED_VALIDATION,
      "RESUME_FROM_CHECKPOINT": RESUME_FROM_CHECKPOINT})


In [ ]:
command = [sys.executable, str(TRAIN_SCRIPT),
           "--config", str(RUN_CONFIG), "--check-config"]
if RESUME_FROM_CHECKPOINT:
    command += ["--resume-from-checkpoint", RESUME_FROM_CHECKPOINT]
subprocess.run(command, check=True)
print("Configuration/data check passed. No training was started; use Step 5 for training.")


## Step 4 — Pretrained baseline on Option B validation

Evaluate the original pretrained model at fixed threshold 0.40. Complete matching
saved results are reused; partial or incompatible files are preserved and rejected.


In [ ]:
if RUN_OPTIONAL_ANALYSES:
    baseline_metrics = evaluate_fixed_or_load(BASE_MODEL, VAL_DATA, BASELINE_OUT)
    display_result("PRETRAINED BASELINE — OPTION B VALIDATION @ 0.40", baseline_metrics)
else:
    print("Validation baseline skipped: RUN_OPTIONAL_ANALYSES=False")


## Step 5 — Full fine-tuning or resume (explicit opt-in)

Set `RUN_TRAINING=True` in Step 3 when ready. For an interrupted run, also set `RESUME_FROM_CHECKPOINT="latest"`. Resume restores the full state and continues toward the same total of 8 epochs; it does not add 8 more epochs. Config and dataset hashes must match. An existing run cannot be silently restarted. Incomplete checkpoints are ignored. Progress since the last complete checkpoint may be repeated.


In [ ]:
if RUN_TRAINING:
    from datetime import datetime, timezone
    command = [sys.executable, "-u", str(TRAIN_SCRIPT),
               "--config", str(RUN_CONFIG)]
    if RESUME_FROM_CHECKPOINT:
        command += ["--resume-from-checkpoint", RESUME_FROM_CHECKPOINT]
    print("Starting training:", " ".join(command), flush=True)
    log_dir = RUN_ROOT / "logs"
    log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / (datetime.now(timezone.utc).strftime("train-%Y%m%dT%H%M%S%fZ") + ".log")
    with log_path.open("x", encoding="utf-8") as log:
        with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end="")
                log.write(line)
                log.flush()
            returncode = process.wait()
        if returncode:
            raise subprocess.CalledProcessError(returncode, command)
else:
    print("Training skipped: RUN_TRAINING=False. Set it to True in Step 3, rerun the config check, then run this cell when ready.")


In [ ]:
if TRAIN_OUT.exists():
    print("Persistent artifacts:", sorted(p.name for p in TRAIN_OUT.iterdir()))
else:
    print(f"Training output directory missing: {TRAIN_OUT}. No training artifacts are available here yet.")


In [ ]:
if (TRAIN_OUT / "training_history.json").exists():
    history = load_json(TRAIN_OUT / "training_history.json")

    provider_label = LABEL_MAPPING["provider"]

    for row in history:
        if "eval_f1" not in row:
            continue

        provider_recall = None
        per_label = row.get("eval_exact_metrics", {}).get("per_label", {})
        if provider_label in per_label:
            provider_recall = per_label[provider_label].get("recall")

        print(
            f"step={row.get('step')} | "
            f"epoch={row.get('epoch')} | "
            f"F1={row['eval_f1']:.4f} | "
            f"P={row['eval_precision']:.4f} | "
            f"R={row['eval_recall']:.4f} | "
            f"PROVIDER_R={provider_recall}"
        )
else:
    print(f"Training history missing: {TRAIN_OUT / 'training_history.json'}. Start or resume training explicitly to generate it.")


## Step 6 — Inspect the best export and optionally evaluate saved validation

Selection occurs during training at every 100 steps and at the final step. `best_model.json` records the selected checkpoint and F1. The exported directory contains GLiNER weights, configuration, tokenizer and selection metadata. No epoch re-ranking is needed.


In [ ]:
selection_path = TRAIN_OUT / "best_model.json"
if selection_path.is_file():
    print(json.dumps(load_json(selection_path), indent=2))
else:
    print("Best-model selection record missing:", selection_path)
if RUN_SAVED_VALIDATION and BEST_MODEL_PATH.is_dir():
    saved_validation = evaluate_fixed_or_load(
        BEST_MODEL_PATH, VAL_DATA, MODEL_EVAL_OUT / BEST_MODEL_NAME)
    display_result("FINE-TUNED — OPTION B VALIDATION @ 0.40", saved_validation)
elif not RUN_SAVED_VALIDATION:
    print("Saved-model validation skipped: RUN_SAVED_VALIDATION=False")
else:
    print("Complete training/export first:", BEST_MODEL_PATH)


In [ ]:
metric_files = [TRAIN_OUT / name for name in ("validation_metrics.json", "train_loss.json")]
if all(path.is_file() for path in metric_files):
    import pandas as pd
    display(pd.DataFrame(load_json(TRAIN_OUT / "validation_metrics.json"))[
        ["step", "epoch", "eval_precision", "eval_recall", "eval_f1"]])
    display(pd.DataFrame(load_json(TRAIN_OUT / "train_loss.json")))
else:
    print("Training metric files missing:", [str(path) for path in metric_files if not path.is_file()])


## Step 7 — Pretrained vs fine-tuned validation comparison

Both models use Option B validation, the same 11 labels, exact-span matching and
fixed threshold 0.40. Differences are fine-tuned minus pretrained.


In [ ]:
if RUN_OPTIONAL_ANALYSES:
    directories = (BASELINE_OUT, MODEL_EVAL_OUT / BEST_MODEL_NAME)
    if all(check_result_files(directory) for directory in directories):
        baseline_metrics = verify_evaluation(BASELINE_OUT,
            expected_evaluation(BASE_MODEL, VAL_DATA, 0.40))
        saved_validation = verify_evaluation(MODEL_EVAL_OUT / BEST_MODEL_NAME,
            expected_evaluation(BEST_MODEL_PATH, VAL_DATA, 0.40))
        compare_results(baseline_metrics, saved_validation, COMPARISON_OUT,
                        "pretrained", "fine_tuned", BASELINE_OUT / "metrics.json",
                        MODEL_EVAL_OUT / BEST_MODEL_NAME / "metrics.json")
    else:
        print("Validation comparison unavailable; run Steps 4 and 6 first.")
else:
    print("Validation comparison skipped: RUN_OPTIONAL_ANALYSES=False")


## Step 8 — Per-label thresholds selected only on Option B validation

Reuse the original 0.10–0.90 grid (step 0.05), candidate decoding at 0.10,
post-decoding filtering, tie rules and PROVIDER recall >= 0.70 constraint. Selected
threshold values may differ from the old experiment. No test tuning is performed.
If calibration fails, its sweep is preserved, the failure is reported, and Step 10's
fixed-threshold evaluations remain available. Partial outputs require a fresh run
directory; they are never deleted or overwritten automatically.


In [ ]:
CALIBRATION_AVAILABLE = False
threshold_names = ("threshold_sweep.csv", "best_thresholds.json", "calibrated_metrics.json",
                   "calibrated_per_label_metrics.csv")
if RUN_OPTIONAL_ANALYSES and BEST_MODEL_PATH.is_dir():
    try:
        targets = [THRESHOLD_OUT / name for name in threshold_names]
        if any(path.exists() for path in targets):
            if not all(path.is_file() for path in targets):
                raise FileExistsError("Partial calibration results preserved; use a fresh run directory.")
        else:
            subprocess.run([sys.executable, str(SCRIPT_ROOT / "tune_per_label_thresholds.py"),
                            "--label-schema", MAPPED_SCHEMA, "--model", str(BEST_MODEL_PATH),
                            "--data", str(VAL_DATA), "--batch-size", "8", "--device", "auto",
                            "--output-dir", str(THRESHOLD_OUT)], check=True)
        selected, candidate, threshold_hash, metadata = validate_frozen_thresholds(
            THRESHOLD_OUT / "best_thresholds.json", BEST_MODEL_PATH, VAL_DATA, TEST_DATA)
        calibrated = load_json(THRESHOLD_OUT / "calibrated_metrics.json")
        if calibrated.get("metadata") != metadata or calibrated.get("thresholds") != selected:
            raise ValueError("Calibrated metrics do not match saved threshold selection")
        CALIBRATION_AVAILABLE = True
        print("Selected validation thresholds:", json.dumps(selected, indent=2))
        display_result("CALIBRATED VALIDATION — TUNING SCORES", calibrated)
    except (subprocess.CalledProcessError, OSError, ValueError, KeyError) as exc:
        print(f"CALIBRATION UNAVAILABLE: {exc}")
        print("No thresholds will be substituted. Fixed 0.40 V4 evaluation remains independent.")
else:
    print("Threshold tuning skipped: enable analyses and complete best-model export first.")


In [ ]:
import pandas as pd
from IPython.display import display
fixed_path = MODEL_EVAL_OUT / BEST_MODEL_NAME / "metrics.json"
tuned_path = THRESHOLD_OUT / "calibrated_metrics.json"
if RUN_OPTIONAL_ANALYSES and CALIBRATION_AVAILABLE and fixed_path.is_file() and tuned_path.is_file():
    fixed = verify_evaluation(MODEL_EVAL_OUT / BEST_MODEL_NAME,
                             expected_evaluation(BEST_MODEL_PATH, VAL_DATA, 0.40))
    tuned = load_json(tuned_path)
    display(pd.DataFrame([{"Evaluation": name, **result["overall"]} for name, result in
            (("Fine-tuned (Fixed 0.40)", fixed), ("Fine-tuned (Tuned Thresholds)", tuned))]))
else:
    print("Fixed vs tuned validation display unavailable; complete Steps 6 and 8 first.")


## Step 9 — Saved provenance and independent model reload

The training directory records resolved config, train/validation hashes, histories,
full resumable checkpoints and best-model selection. Logs are under this run's
`logs/`. The independent reload cell needs only mounted Drive and installed
dependencies; it does not load datasets or depend on prior notebook variables.


In [ ]:
for name in ("training_config.json", "data_manifest.json", "best_model.json"):
    path = TRAIN_OUT / name
    if path.exists():
        print(name)
        print(json.dumps(load_json(path), indent=2))
    else:
        print(f"Run artifact missing: {path}. Training/export may not have completed yet.")


In [ ]:
from pathlib import Path
import torch
from gliner import GLiNER

SAVED_MODEL = Path("/content/drive/MyDrive/Holokai_Project/week8/Final_model_Option_B/training/best_model")
if SAVED_MODEL.is_dir():
    inference_model = GLiNER.from_pretrained(str(SAVED_MODEL), local_files_only=True)
    inference_model.to("cuda" if torch.cuda.is_available() else "cpu")
    inference_model.eval()
    print("Independently reloaded:", SAVED_MODEL)
    # Release the diagnostic model so it cannot occupy GPU memory during Step 10.
    del inference_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    print("Complete training/export before reloading:", SAVED_MODEL)


## Step 10 — Frozen-model Option B V4 test: baseline and fine-tuned

In a fresh runtime, mount Drive and run the dependency installation cell, then this
section. It defines its own paths, helpers and execution switches. Set
`RUN_V4_EVALUATION=True` only after the annotation policy, model and validation
choices are frozen. Baseline and fine-tuned fixed-0.40 evaluations do not require a
threshold file. The following cell separately applies validation-selected per-label
thresholds if available; it never searches on V4 or applies fine-tuned thresholds
to the pretrained baseline. Complete compatible results are reused.


In [ ]:
from pathlib import Path
from collections import Counter
import hashlib
import importlib
import json
import subprocess
import sys

PROJECT = Path("/content/drive/MyDrive/Holokai_Project")
DATA_ROOT = PROJECT / "week8/Holokai_Option_B_Annotation_Files"
RUN_ROOT = PROJECT / "week8/Final_model_Option_B"
TRAIN_DATA = DATA_ROOT / "train_gliner_mapping_option_B.json"
VAL_DATA = DATA_ROOT / "val_gliner_mapping_option_B.json"
TEST_DATA = DATA_ROOT / "corpus_V4_gliner_mapping_option_B.json"
RUN_CONFIG = RUN_ROOT / "configs/finetune_final_run_option_b.json"
TRAIN_OUT = RUN_ROOT / "training"
BASELINE_OUT = RUN_ROOT / "baseline"
MODEL_EVAL_OUT = RUN_ROOT / "model_eval"
COMPARISON_OUT = RUN_ROOT / "comparison"
THRESHOLD_OUT = RUN_ROOT / "threshold_tuning"
BEST_MODEL_NAME = "best_model"
BEST_MODEL_PATH = TRAIN_OUT / BEST_MODEL_NAME
BASE_MODEL = "knowledgator/gliner-pii-base-v1.0"
MAPPED_SCHEMA = "recommended"
REFERENCE_CONFIG = {'label_schema': 'recommended', 'base_model': 'knowledgator/gliner-pii-base-v1.0', 'train_data': '/content/drive/MyDrive/Holokai_Project/finetune/data/train_gliner_mapping.json', 'val_data': '/content/drive/MyDrive/Holokai_Project/finetune/data/val_gliner_mapping.json', 'output_dir': '/content/drive/MyDrive/Holokai_Project/final_model/outputs/training', 'learning_rate': 0.00012, 'batch_size': 16, 'eval_batch_size': 8, 'epochs': 8, 'warmup_ratio': 0.1, 'weight_decay': 0.1, 'seed': 42, 'eval_steps': 100, 'logging_steps': 10, 'threshold': 0.4, 'flat_ner': False, 'gradient_accumulation_steps': 1, 'max_grad_norm': 1.0, 'adam_beta1': 0.9, 'adam_beta2': 0.999, 'adam_epsilon': 1e-08, 'lr_scheduler_type': 'linear', 'loss_reduction': 'mean', 'device': 'auto', 'save_best_model': True}
SUPPORT_HASHES = {'finetune/requirements-colab.txt': '8109cb2ec9815906b0e178f9687f7aca6eab59b5573df33524497a1c3f2ce3e8', 'finetune/scripts/train_gliner_final_run.py': '6c38397ff4585b6c245d77786f4a0f851a4c7bc7059ab636bdc1e61772d3b107', 'finetune/scripts/evaluate_gliner.py': 'dbf305c3cc14a149291cb2715804a6ea56b3ee4072cd8575266f623ed1fe1300', 'finetune/scripts/label_schemas.py': '871d28532ec98e4130ae7e5d0b4f34c2ea1e7b8b63b5455da6fbf3d76fcb7b47', 'finetune/scripts/validate_gliner_data.py': 'c6d0536f8692199da3b42d6b4f156481f3f6b1f0ea53386364ed006657021667', 'finetune/scripts/tune_per_label_thresholds.py': '9b6f092508bf9a5b0ec81f35c885f6f39f2049eb2fd22c3679cb01f508fca06b'}

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def resolve_support():
    """Reuse identical scripts; otherwise extract an isolated, verified copy."""
    import zipfile
    def matches(root):
        return all((root / name).is_file() and sha256_file(root / name) == digest
                   for name, digest in SUPPORT_HASHES.items())
    if matches(PROJECT):
        return PROJECT
    support = RUN_ROOT / "support"
    if matches(support):
        return support
    archive_path = RUN_ROOT / "option_b_colab_support.zip"
    if not archive_path.is_file():
        raise FileNotFoundError(
            f"Shared scripts are missing or differ from the original run. Upload "
            f"option_b_colab_support.zip to {RUN_ROOT}; original files will be preserved.")
    with zipfile.ZipFile(archive_path) as archive:
        for name, digest in SUPPORT_HASHES.items():
            raw = archive.read(name)
            if hashlib.sha256(raw).hexdigest() != digest:
                raise ValueError(f"Support archive checksum mismatch: {name}")
            target = support / name
            if target.exists():
                if not target.is_file() or sha256_file(target) != digest:
                    raise FileExistsError(f"Incompatible support file preserved: {target}")
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with target.open("xb") as handle:
                    handle.write(raw)
    assert matches(support)
    return support

def import_run_modules(support):
    scripts = support / "finetune/scripts"
    sys.path.insert(0, str(scripts))
    names = ("label_schemas", "validate_gliner_data", "evaluate_gliner",
             "train_gliner_final_run", "tune_per_label_thresholds")
    # Invalidate the entire import group if one member belongs to another run.
    if any(name in sys.modules and Path(sys.modules[name].__file__).resolve() !=
           (scripts / f"{name}.py").resolve() for name in names):
        for name in names:
            sys.modules.pop(name, None)
    modules = {name: importlib.import_module(name) for name in names}
    for name, module in modules.items():
        if Path(module.__file__).resolve() != (scripts / f"{name}.py").resolve():
            raise ImportError(f"Unexpected script loaded: {module.__file__}")
    return scripts, modules

def load_json(path):
    from validate_gliner_data import unique_keys, reject_constant
    return json.loads(Path(path).read_text(encoding="utf-8"),
                      object_pairs_hook=unique_keys, parse_constant=reject_constant)

def preserve_json(path, value):
    """Reuse an identical result, never replace an existing one."""
    path = Path(path)
    if path.exists():
        if load_json(path) != value:
            raise FileExistsError(f"Different saved result preserved: {path}")
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("x", encoding="utf-8") as handle:
        json.dump(value, handle, indent=2, ensure_ascii=False, allow_nan=False)
        handle.write("\n")

def preserve_csv(path, frame):
    import pandas as pd
    path = Path(path)
    if path.exists():
        pd.testing.assert_frame_equal(pd.read_csv(path), frame.reset_index(drop=True),
                                      check_dtype=False)
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False, mode="x")

def check_result_files(output):
    files = [Path(output) / name for name in
             ("metrics.json", "per_label_metrics.csv", "predictions.json")]
    if any(path.exists() for path in files):
        if not all(path.is_file() for path in files):
            raise FileExistsError(f"Partial results preserved at {output}; use a new directory.")
        return True
    return False

def expected_evaluation(model_name, data_path, threshold):
    from evaluate_gliner import load_data
    rows, digest = load_data(Path(data_path), MAPPED_SCHEMA)
    return {"model": str(model_name), "data_sha256": digest,
            "sample_count": len(rows), "label_schema": MAPPED_SCHEMA,
            "batch_size": 8, "flat_ner": False, "multi_label": False,
            "threshold": threshold, "averaging": "micro",
            "matching": "exact inclusive token start/end + exact label within sample"}

def verify_evaluation(output, expected):
    from label_schemas import get_labels
    if not check_result_files(output):
        raise FileNotFoundError(f"Evaluation results missing: {output}")
    result = load_json(Path(output) / "metrics.json")
    for key, value in expected.items():
        if result.get("metadata", {}).get(key) != value:
            raise ValueError(f"Incompatible {key} in {output}; existing files preserved.")
    if set(result["per_label"]) != set(get_labels(MAPPED_SCHEMA)):
        raise ValueError(f"Unexpected result labels: {output}")
    return result

def evaluate_fixed_or_load(model_name, data_path, output):
    expected = expected_evaluation(model_name, data_path, 0.40)
    if not check_result_files(output):
        subprocess.run([sys.executable, str(SCRIPT_ROOT / "evaluate_gliner.py"),
                        "--label-schema", MAPPED_SCHEMA, "--model", str(model_name),
                        "--data", str(data_path), "--threshold", "0.40", "--batch-size", "8",
                        "--output-dir", str(output), "--device", "auto"], check=True)
    result = verify_evaluation(output, expected)
    print("Results available:", output)
    return result

def validate_frozen_thresholds(path, model_path, validation_path, test_path):
    from evaluate_gliner import load_selected_thresholds
    selected, candidate, digest, metadata = load_selected_thresholds(
        path, MAPPED_SCHEMA, False, str(model_path))
    if metadata.get("data_sha256") != sha256_file(validation_path):
        raise ValueError("Thresholds were not selected on the current Option B validation file.")
    if metadata["data_sha256"] == sha256_file(test_path):
        raise ValueError("Test data cannot be the threshold-selection data.")
    return selected, candidate, digest, metadata

def display_result(title, result):
    import pandas as pd
    from IPython.display import display
    fields = ["precision", "recall", "f1", "tp", "fp", "fn", "support"]
    fmt = {name: "{:.2%}" for name in ("precision", "recall", "f1")}
    print(title)
    display(pd.DataFrame([result["overall"]])[fields].style.format(fmt))
    table = pd.DataFrame.from_dict(result["per_label"], orient="index")[fields]
    table.index.name = "label"
    table.insert(0, "threshold", [result.get("thresholds", {}).get(label, 0.40)
                                  for label in table.index])
    display(table.style.format(fmt))

def compare_results(left, right, output, left_name, right_name, left_path, right_path):
    import pandas as pd
    from IPython.display import display
    for key in ("data_sha256", "sample_count", "label_schema", "batch_size", "flat_ner",
                "multi_label", "matching", "averaging"):
        if left["metadata"][key] != right["metadata"][key]:
            raise ValueError(f"Comparison evaluation mismatch: {key}")
    if left["overall"]["support"] != right["overall"]["support"]:
        raise ValueError("Comparison gold support mismatch")
    fields = ["precision", "recall", "f1", "tp", "fp", "fn", "support"]
    overall = [{"metric": field, left_name: left["overall"][field],
                right_name: right["overall"][field],
                "change_right_minus_left": right["overall"][field] - left["overall"][field]}
               for field in fields]
    per_label = []
    for label in left["per_label"]:
        row = {"label": label}
        if left["per_label"][label]["support"] != right["per_label"][label]["support"]:
            raise ValueError(f"Comparison per-label support mismatch: {label}")
        for field in fields:
            a, b = left["per_label"][label][field], right["per_label"][label][field]
            row.update({f"{field}_{left_name}": a, f"{field}_{right_name}": b,
                        f"{field}_delta": b - a})
        per_label.append(row)
    record = {"left_result": str(left_path), "right_result": str(right_path),
              "difference": f"{right_name} - {left_name}",
              "data_sha256": left["metadata"]["data_sha256"],
              "thresholds_sha256": right["metadata"].get("thresholds_sha256"),
              "comparison": overall, "per_label_comparison": per_label}
    output = Path(output)
    preserve_json(output / "overall_comparison.json", record)
    preserve_csv(output / "overall_comparison.csv", pd.DataFrame(overall))
    preserve_csv(output / "per_label_comparison.csv", pd.DataFrame(per_label))
    print(f"Change = {right_name} - {left_name}; rate changes are absolute proportions.")
    display(pd.DataFrame(overall))
    display(pd.DataFrame(per_label))
    print("Comparison saved:", output)


RUN_V4_EVALUATION = True
SUPPORT_ROOT = resolve_support()
SCRIPT_ROOT, modules = import_run_modules(SUPPORT_ROOT)
evaluator = modules["evaluate_gliner"]
tuner = modules["tune_per_label_thresholds"]
v4_baseline_out = RUN_ROOT / "v4_test_baseline"
v4_fixed_out = RUN_ROOT / "v4_test"
v4_per_label_out = RUN_ROOT / "v4_test_per_label"
threshold_file = THRESHOLD_OUT / "best_thresholds.json"

if RUN_V4_EVALUATION:
    # Do not use any V4 metrics to revisit model or threshold choices.
    if not BEST_MODEL_PATH.is_dir() or not (TRAIN_OUT / "best_model.json").is_file():
        raise FileNotFoundError("Complete the Option B best-model export before test evaluation.")
    training_manifest = load_json(TRAIN_OUT / "data_manifest.json")
    for split, path in (("train", TRAIN_DATA), ("val", VAL_DATA)):
        if training_manifest[split]["sha256"] != sha256_file(path):
            raise ValueError(f"Frozen model used a different {split} dataset")
    if load_json(TRAIN_OUT / "training_config.json") != modules["train_gliner_final_run"].load_config(RUN_CONFIG):
        raise ValueError("Frozen model config differs from the current Option B config")
    v4_baseline_metrics = evaluate_fixed_or_load(BASE_MODEL, TEST_DATA, v4_baseline_out)
    fixed_v4_metrics = evaluate_fixed_or_load(BEST_MODEL_PATH, TEST_DATA, v4_fixed_out)
else:
    print("V4 evaluation disabled. Freeze the model/validation choices, then enable this cell.")


### Apply frozen validation-selected per-label thresholds

Uses the original evaluator and tuner functions, including the shared prediction
pool at candidate threshold 0.10. The threshold hash, validation hash and model
path are checked before inference and when reusing saved results.


In [ ]:
V4_PER_LABEL_AVAILABLE = True
if RUN_V4_EVALUATION:
    if not threshold_file.is_file():
        print("Per-label V4 evaluation unavailable: no validation-selected thresholds.")
        print("Baseline and fine-tuned fixed-0.40 results remain available.")
    else:
        # Invalid or stale thresholds fail explicitly; never silently substitute another file.
        selected, candidate, threshold_hash, selection_metadata = validate_frozen_thresholds(
            threshold_file, BEST_MODEL_PATH, VAL_DATA, TEST_DATA)
        expected = {**expected_evaluation(BEST_MODEL_PATH, TEST_DATA, None),
                    "thresholds_sha256": threshold_hash, "candidate_threshold": candidate}
        if not check_result_files(v4_per_label_out):
            from datetime import datetime, timezone
            from importlib.metadata import version
            import torch
            from gliner import GLiNER
            from label_schemas import schema_metadata
            test_data, test_hash = evaluator.load_data(TEST_DATA, MAPPED_SCHEMA)
            model = GLiNER.from_pretrained(str(BEST_MODEL_PATH), local_files_only=True)
            device = "cuda" if torch.cuda.is_available() else "cpu"
            model.to(device)
            model.eval()
            try:
                with torch.inference_mode():
                    pool = evaluator.predict(model, test_data, candidate, 8, False, MAPPED_SCHEMA)
                filtered = tuner.filter_predictions(pool, selected, MAPPED_SCHEMA)
                scores = evaluator.compute_metrics([row["ner"] for row in test_data],
                    [[(e["start"], e["end"], e["label"]) for e in row["predicted_entities"]]
                     for row in filtered], label_schema=MAPPED_SCHEMA)
                scores["thresholds"] = selected
                scores["metadata"] = {
                    **expected, "data": str(TEST_DATA), **schema_metadata(MAPPED_SCHEMA),
                    "device": device, "zero_division": 0,
                    "duplicate_predictions": "extra occurrences count as FP",
                    "gliner_version": version("gliner"), "torch_version": torch.__version__,
                    "created_at_utc": datetime.now(timezone.utc).isoformat(),
                    "thresholds_file": str(threshold_file.resolve()),
                    "filter_comparison": "score >= threshold",
                    "calibration_method": selection_metadata["calibration_method"],
                    "threshold_selection_metadata": selection_metadata,
                    "evaluation_scope": "held-out evaluation with frozen validation-selected thresholds"}
                # Recheck input provenance before writing any result.
                if test_hash != expected["data_sha256"] or sha256_file(TEST_DATA) != test_hash:
                    raise ValueError("Test data changed during evaluation")
                if sha256_file(threshold_file) != threshold_hash:
                    raise ValueError("Threshold file changed during evaluation")
                evaluator.save_results(v4_per_label_out, scores, filtered)
            finally:
                del model
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
        per_label_v4_metrics = verify_evaluation(v4_per_label_out, expected)
        if per_label_v4_metrics.get("thresholds") != selected:
            raise ValueError("Saved V4 thresholds differ from the frozen validation thresholds")
        V4_PER_LABEL_AVAILABLE = True
else:
    print("Per-label V4 evaluation skipped: RUN_V4_EVALUATION=False")


### Display V4 results and compare pretrained vs fine-tuned at fixed 0.40

The primary model comparison uses the same fixed threshold on both models.
Overall and per-label deltas are fine-tuned minus pretrained. Displays read saved
files, so they also work after the per-label calibration step was skipped or failed.


In [ ]:
if RUN_V4_EVALUATION:
    v4_baseline_metrics = verify_evaluation(v4_baseline_out,
        expected_evaluation(BASE_MODEL, TEST_DATA, 0.40))
    fixed_v4_metrics = verify_evaluation(v4_fixed_out,
        expected_evaluation(BEST_MODEL_PATH, TEST_DATA, 0.40))
    display_result("PRETRAINED BASELINE — OPTION B V4 @ 0.40", v4_baseline_metrics)
    display_result("FINE-TUNED — OPTION B V4 @ 0.40", fixed_v4_metrics)
    compare_results(v4_baseline_metrics, fixed_v4_metrics, RUN_ROOT / "v4_baseline_comparison",
                    "pretrained", "fine_tuned", v4_baseline_out / "metrics.json",
                    v4_fixed_out / "metrics.json")
else:
    print("V4 baseline comparison skipped: RUN_V4_EVALUATION=False")


### Compare fine-tuned fixed vs validation-selected per-label thresholds

Both results must come from the same exported model and V4 data. Deltas are
per-label minus fixed. No model selection or threshold tuning occurs here.


In [ ]:
if RUN_V4_EVALUATION:
    if threshold_file.is_file() and check_result_files(v4_per_label_out):
        selected, candidate, threshold_hash, selection_metadata = validate_frozen_thresholds(
            threshold_file, BEST_MODEL_PATH, VAL_DATA, TEST_DATA)
        per_label_v4_metrics = verify_evaluation(v4_per_label_out,
            {**expected_evaluation(BEST_MODEL_PATH, TEST_DATA, None),
             "thresholds_sha256": threshold_hash, "candidate_threshold": candidate})
        if per_label_v4_metrics.get("thresholds") != selected:
            raise ValueError("Saved per-label thresholds do not match validation selection")
        fixed_v4_metrics = verify_evaluation(v4_fixed_out,
            expected_evaluation(BEST_MODEL_PATH, TEST_DATA, 0.40))
        display_result("FINE-TUNED — OPTION B V4 WITH VALIDATION-SELECTED THRESHOLDS",
                       per_label_v4_metrics)
        compare_results(fixed_v4_metrics, per_label_v4_metrics, RUN_ROOT / "v4_test_comparison",
                        "fixed_0_40", "validation_selected_per_label", v4_fixed_out / "metrics.json",
                        v4_per_label_out / "metrics.json")
    else:
        print("Per-label comparison unavailable; fixed-0.40 baseline comparison is independent.")
else:
    print("V4 threshold comparison skipped: RUN_V4_EVALUATION=False")
